# R3 — Cổng miền áp dụng cho công cụ

**Mục đích:** học hai tầng cổng trả lời *"tile này có thuộc miền `PSI(WT) ≤ 4,0` mà mô hình ΔPSI đã học không?"*:
tầng (i) ánh xạ PSI đo của sàng peptidome GPS → trạng thái miền; tầng (ii) ridge 420 đặc trưng từ trình tự → PSI → trạng thái miền.
Tiêu chí chốt trước: `research/r3-domain-gate-2026-10-22/PROTOCOL.md` (SHA-256 nhúng ở ô 1). Chỉ CPU (vài phút), **không cần GPU**.

**Trước khi chạy:** (1) R0 phải ĐẠT trong cùng loại môi trường (đặt `R0_CONFIRMED = True` ở ô 1); (2) gắn 2 dataset (`degron-code-*`, `degron-data-v1`).
Kết quả: `/kaggle/working/out/r3-results.zip` — giải nén vào `research/r3-domain-gate-2026-10-22/run-<nền-tảng>/` (xem `research/notebook-kit-2026-10-22/RUN.md`).

In [ ]:
# Ô 1 — tham số
PROTOCOL_SHA256 = "8188e3e5e8901e32bbaac5547a9f4bfc16ed2bcce1436291a700fca972b897ff"
R0_CONFIRMED = False       # chỉ đặt True sau khi 00-reproduce.ipynb báo "R0 ĐẠT" ở môi trường này
SMOKE = False              # True = chạy thử nhanh trên 30.000 tile (chỉ kiểm hạ tầng, KHÔNG dùng làm kết quả)
assert R0_CONFIRMED or SMOKE, "Chạy R0 trước (PROTOCOL §4). Đặt R0_CONFIRMED = True sau khi R0 ĐẠT."

In [ ]:
# --- bootstrap (giống hệt ở mọi notebook): tìm gói mã `degron-code*` và nạp nbkit ---
import sys, os, glob, zipfile

def _find_nbkit_dir():
    pats = ['/kaggle/input/**/notebook-kit-2026-10-22/nbkit.py',
            '/content/**/notebook-kit-2026-10-22/nbkit.py']
    for p in pats:
        hit = glob.glob(p, recursive=True)
        if hit:
            return os.path.dirname(hit[0])
    extra = os.environ.get('DEGRON_PACKAGES', '')      # chạy local: thư mục chứa các zip
    zips = (glob.glob(extra + '/degron-code*.zip') if extra else []) + (glob.glob('/content/degron-code*.zip') + glob.glob('/content/drive/MyDrive/**/degron-code*.zip', recursive=True)
            + glob.glob('./degron-code*.zip') + glob.glob('data/processed/packages/degron-code*.zip'))
    if zips:
        dest = '/tmp/degron-code-boot'
        zipfile.ZipFile(zips[0]).extractall(dest)
        return glob.glob(dest + '/**/notebook-kit-2026-10-22', recursive=True)[0]
    raise FileNotFoundError('Không thấy gói mã degron-code (dataset đã gắn vào notebook chưa?) — xem RUN.md bước 3-4')

sys.path.insert(0, _find_nbkit_dir())
import nbkit
print('nbkit:', nbkit.__file__)

In [ ]:
# Ô 3 — ghép cây thư mục, kiểm SHA-256 dữ liệu, kiểm PROTOCOL
ctx = nbkit.setup('R3', protocol='research/r3-domain-gate-2026-10-22/PROTOCOL.md',
                  protocol_sha256=PROTOCOL_SHA256, needs=['R3'], threads=2)

In [ ]:
# Ô 4 — chạy (hàm nằm trong run_r3.py; notebook chỉ gọi). Vài phút CPU.
ctx.add_to_path('r3-domain-gate-2026-10-22')
import run_r3
res = run_r3.run(ctx, smoke=SMOKE)

In [ ]:
# Ô 5 — kết luận theo đúng PROTOCOL §2-§3 (không diễn giải thêm)
t1, ta, tb = res['tier1'], res['tier2_stage_A'], res['tier2_stage_B']
print('TẦNG (i)   :', 'DÙNG ĐƯỢC' if res['usable_tier1'] else 'KHÔNG ĐẠT', t1['verdict'])
print('TẦNG (ii)  :', 'DÙNG ĐƯỢC' if res['usable_tier2'] else 'KHÔNG ĐẠT', tb['verdict'])
print(f"  tầng (i)  AUROC OOF {t1['auroc_oof']:.4f}  BSS {t1['bss_oof']:.3f}  d0 lọt in-domain {t1['frac_d0_in_domain']:.3f}")
print(f"  tầng (ii) Spearman A {ta['spearman_point']:.3f}  AUROC B {tb['auroc_oof']:.4f}  BSS {tb['bss_oof']:.3f}  d0 lọt in-domain {tb['frac_d0_in_domain']:.3f}")
if SMOKE:
    print("\n⚠ SMOKE = True: kết quả này chỉ kiểm hạ tầng, KHÔNG phải kết quả R3.")

In [ ]:
# Ô 6 — ghi gate-weights.npz, gate-card.json, results.json và nén để tải về
run_r3.write_outputs(res, ctx.out)
ctx.finalize()